In [2]:

%%capture
!pip install "git+https://github.com/google-deepmind/ai-foundations.git@main"

# Packages used.
import os # Used for setting Keras configuration variables.
os.environ["KERAS_BACKEND"] = "jax" # Set a parameter for Keras.
import re # Used for splitting text on whitespace.

import keras # Used for defining an training the model.
import pandas as pd # Used for loading the dataset.
import tensorflow as tf # Used for shuffling the dataset.

# Used for displaying nicer error messages.
from IPython.display import display, HTML
from ai_foundations import training # For training your model.
from ai_foundations import generation # For prompting your model.
from ai_foundations import visualizations # For visualizing probabilities.
from ai_foundations.feedback.course_1 import slm # For providing feedback.

# The following line provides configuration for Keras.
keras.utils.set_random_seed(812)  # For Keras layers.

In [3]:
africa_galore = pd.read_json(
    "https://storage.googleapis.com/dm-educational/assets/ai_foundations/africa_galore.json"
)
dataset = africa_galore["description"].values
print("Loaded dataset with", dataset.shape[0], "paragraphs.")

Loaded dataset with 232 paragraphs.


In [4]:
class SimpleWordTokenizer:
    """A simple word tokenizer.

    The tokenizer splits the text sequence based on whitespace, using the
    `encode` method to convert the text into a sequence of indices and the
    `decode` method to convert indices back into text.

    The simple word tokenizer that can be initialized with a corpus or using a
    provided vocabulary list

    Typical usage example:

        corpus = "Hello there!"
        tokenizer = SimpleWordTokenizer(text)
        print(tokenizer.encode('Hello'))

    """

    # Define constants.
    UNKNOWN_TOKEN = ""
    PAD_TOKEN = ""

    def __init__(self, corpus: list[str], vocabulary: list[str] | None = None):
        """Initializes the tokenizer with texts in corpus or with a vocabulary.

        Args:
          corpus: Input text dataset.
          vocabulary: A pre-defined vocabulary. If None,
              the vocabulary is automatically inferred from the texts.
        """

        if vocabulary is None:
            # Build the vocabulary from scratch.
            if isinstance(corpus, str):
                corpus = [corpus]

            # Convert text sequence to tokens.
            tokens = []
            for text in corpus:
                for token in self.space_tokenize(text):
                    tokens.append(token)

            # Create a vocabulary comprising of unique tokens.
            vocabulary = self.build_vocabulary(tokens)

            # Add special unknown and pad tokens to the vocabulary list.
            self.vocabulary = (
                [self.PAD_TOKEN] + vocabulary + [self.UNKNOWN_TOKEN]
            )

        else:
            self.vocabulary = vocabulary

        # Size of vocabulary.
        self.vocabulary_size = len(self.vocabulary)

        # Create token-to-index and index-to-token mappings.
        self.token_to_index = {}
        self.index_to_token = {}
        # Loop through all tokens in the vocabulary. enumerate automatically
        # assigns a unique index to each token.
        for index, token in enumerate(self.vocabulary):
            self.token_to_index[token] = index
            self.index_to_token[index] = token

        # Map the special tokens to their IDs.
        self.pad_token_id = self.token_to_index[self.PAD_TOKEN]
        self.unknown_token_id = self.token_to_index[self.UNKNOWN_TOKEN]

    def space_tokenize(self, text: str) -> list[str]:
        """Splits a given text on whitespace into tokens.

        Args:
            text: Text to split on whitespace.

        Returns:
            List of tokens after splitting `text`.
        """

        # Use re.split such that multiple spaces are treated as a single
        # separator.
        return re.split(" +", text)

    def join_text(self, text_list: list[str]) -> str:
        """Combines a list of tokens into a single string.

        The combined tokens, as a single string, are separated by spaces in the
        string.

        Args:
            text_list: List of tokens to be joined.

        Returns:
            String with all tokens joined with a whitespace.

        """
        return " ".join(text_list)

    def build_vocabulary(self, tokens: list[str]) -> list[str]:
        """Create a vocabulary list from the list of tokens.

        Args:
            tokens: The list of tokens in the dataset.

        Returns:
            List of unique tokens (vocabulary) in the dataset.
        """
        return sorted(list(set(tokens)))

    def encode(self, text: str) -> list[int]:
        """Encodes a text sequence into a list of indices.

        Args:
            text: The input text to be encoded.

        Returns:
            A list of indices corresponding to the tokens in the input text.
        """

        # Convert tokens into indices.
        indices = []
        unk_index = self.token_to_index[self.UNKNOWN_TOKEN]
        for token in self.space_tokenize(text):
            token_index = self.token_to_index.get(token, unk_index)
            indices.append(token_index)

        return indices

    def decode(self, indices: int | list[int]) -> str:
        """Decodes a list (or single index) of integers back into tokens.

        Args:
            indices: A single index or a list of indices to be
                decoded into tokens.

        Returns:
            A string of decoded tokens corresponding to the input indices.
        """

        # If a single integer is passed, convert it into a list.
        if isinstance(indices, int):
            indices = [indices]

        # Map indices to tokens.
        tokens = []
        for index in indices:
            token = self.index_to_token.get(index, self.unknown_token_id)
            tokens.append(token)

        # Join the decoded tokens into a single string.
        return self.join_text(tokens)


# Initialize the tokenizer. This will build the tokenizer's vocabulary with
# all the tokens that appear in the dataset.
tokenizer = SimpleWordTokenizer(dataset)

# Translate all tokens to their corresponding IDs.
encoded_tokens = []
for text in dataset:
    # Split text into tokens and translate the tokens to token IDs.
    token_ids = tokenizer.encode(text)
    encoded_tokens.append(token_ids)


In [5]:

encoded_tokens[0][:10]

[814, 511, 985, 5092, 4802, 5183, 2800, 1363, 4792, 2134]

In [6]:

print(f"Length of first paragraph: {len(encoded_tokens[0]):,}")

Length of first paragraph: 118


In [7]:
# Add your code to compute the length of the shortest paragraph here.
shortest_paragraph_length = len(min(encoded_tokens, key = len))

# Add your code to compute the length of the longest paragraph here.
longest_paragraph_length = len(max(encoded_tokens, key = len))

print(f"Length of the shortest paragraph is:", shortest_paragraph_length)
print(f"Length of the longest paragraph is:", longest_paragraph_length)

Length of the shortest paragraph is: 26
Length of the longest paragraph is: 318


In [8]:
# @title Run this cell to test your code

slm.test_max_min_seqlen(
    shortest_paragraph_length, longest_paragraph_length, encoded_tokens
)

✅ Nice! Your answer looks correct.


In [9]:
# @title Set `max_length` for padding and truncating data.

max_length = 300  # @param {type: "number"}

if max_length <= 0:
    display(
        HTML(
            f"Error:Max length must be greater than 0. Please"
            f" increase max_length."
        )
    )

elif max_length > longest_paragraph_length:
    display(
        HTML(
            f"Error:The padding token "
            f" {tokenizer.pad_token_id} will be added to all"
            f" sequences - you probably don't want that. Please reduce"
            f" max_length."
        )
    )

else:
    if max_length < longest_paragraph_length:
        display(
            HTML(
                f"Note: The longest paragraph has"
                f" {longest_paragraph_length} tokens,"
                f" but max_length is set to {max_length}."
                f" Paragraphs longer than max_length will be"
                " truncated."
            )
        )

    padded_sequences = keras.preprocessing.sequence.pad_sequences(
        encoded_tokens,
        maxlen=max_length,
        padding="post",
        truncating="post",
        value=tokenizer.pad_token_id,
    )

    print("New length of first paragraph:", len(padded_sequences[0]), "\n")

    print(
        "Padding makes the length of all sequences the same as the specified"
        " `max_length`."
    )

    print(
        "Notice the padded token IDs {tokenizer.pad_token_id} appearing at the"
        f" end of the sequence.\n"
    )
    print("Padded tokens of first paragraph:\n", padded_sequences[0])

New length of first paragraph: 300 

Padding makes the length of all sequences the same as the specified `max_length`.
Notice the padded token IDs {tokenizer.pad_token_id} appearing at the end of the sequence.

Padded tokens of first paragraph:
 [ 814  511  985 5092 4802 5183 2800 1363 4792 2134 2856 4792 1584 5092
 2088  814 1134 3043 2922  912 2821  170 2623 4792 2023 3807 3576  912
 1653 3772 4792 2775 1244  912 4409 3280 1030 4792 1158 3049 1992  912
 1868 2486 2437  135 5189 3422  445 3388 2078 4849 4792 3407 2706 1259
 4692 2856 4839 5183 4792 4078  814 3406 4259 4849 2389 4831 2707  912
 3821 1829 3522 2134 1030 2955  185 1076 2707 3683 5143 1849 4343 1030
 1546 1446 4983 2856 4792 2876 4078  814 3406 5092 3366 4788 2968 2151
 2938 5092  912 1450 3522 3101  912 1672 4849 4793 4295 2721  912 5036
 2224 3522 4792 4437 3522  513 5261 5261 5261 5261 5261 5261 5261 5261
 5261 5261 5261 5261 5261 5261 5261 5261 5261 5261 5261 5261 5261 5261
 5261 5261 5261 5261 5261 5261 5261 5261 526

In [10]:

# Prepare input and target for the transformer model.
# For each example, extract all tokens except the last one.
input_sequences = padded_sequences[:, :-1]
# For each example, extract all tokens except the first one.
target_sequences = padded_sequences[:, 1:]

In [11]:

print("First 10 token IDs in first input sequence:", input_sequences[0, :10])
print(
    "First 10 tokens in first input sequence:",
    tokenizer.decode(input_sequences[0, :10]),
)

print("\n")

print("First 10 token IDs in first target sequence:", target_sequences[0, :10])
print(
    "First 10 tokens in target sequence:",
    tokenizer.decode(target_sequences[0, :10])
)

First 10 token IDs in first input sequence: [ 814  511  985 5092 4802 5183 2800 1363 4792 2134]
First 10 tokens in first input sequence: The Lagos air was thick with humidity, but the energy


First 10 token IDs in first target sequence: [ 511  985 5092 4802 5183 2800 1363 4792 2134 2856]
First 10 tokens in target sequence: Lagos air was thick with humidity, but the energy in


In [12]:
max_length = input_sequences.shape[1]

In [13]:
# Create TensorFlow dataset to prepare sequences.
tf_dataset = tf.data.Dataset.from_tensor_slices((input_sequences, target_sequences))

# Randomly shuffle the dataset.
# The buffer_size determines how many examples from the dataset
# are held in memory before shuffling.
# If you are working with a very large dataset,
# reduce the buffer_size as needed.
tf_dataset = tf_dataset.shuffle(buffer_size=len(input_sequences))

# Specify batch size.
batch_size = 32  # @param {type: "number"}

# Create batches.
batches = tf_dataset.batch(batch_size)

for batch in batches.take(1):
    print(batch)

(<tf.Tensor: shape=(32, 299), dtype=int32, numpy=
array([[ 719, 5092, 4815, ..., 5261, 5261, 5261],
       [ 797,  597,  912, ..., 5261, 5261, 5261],
       [ 470, 4084, 2932, ..., 5261, 5261, 5261],
       ...,
       [ 814, 4079, 1171, ..., 5261, 5261, 5261],
       [ 814, 3085, 2932, ..., 5261, 5261, 5261],
       [ 358, 1605, 2935, ..., 5261, 5261, 5261]], dtype=int32)>, <tf.Tensor: shape=(32, 299), dtype=int32, numpy=
array([[5092, 4815, 4403, ..., 5261, 5261, 5261],
       [ 597,  912, 2364, ..., 5261, 5261, 5261],
       [4084, 2932,  912, ..., 5261, 5261, 5261],
       ...,
       [4079, 1171, 3522, ..., 5261, 5261, 5261],
       [3085, 2932, 4792, ..., 5261, 5261, 5261],
       [1605, 2935, 2968, ..., 5261, 5261, 5261]], dtype=int32)>)


In [14]:
model = training.create_model(
    max_length=max_length,
    vocabulary_size=tokenizer.vocabulary_size,
    learning_rate=1e-4
)


In [15]:
prompt = "Abeni,"
prompt_ids = tokenizer.encode(prompt)
text_gen_callback = training.TextGenerator(
    max_tokens=10, start_tokens=prompt_ids, tokenizer=tokenizer, print_every=10
)

In [ ]:

num_epochs = 200  # @param {type: "number"}
# verbose=2: Instructs the model.fit method to print one line per
# epoch so you see how the loss is decreasing and generated texts improving.
history = model.fit(
    x=batches, verbose=2, epochs=num_epochs, callbacks=[text_gen_callback]
)


Epoch 1/200


In [ ]:

prompt = "Abeni, a bright-eyed" #@param {type: "string"}
num_tokens_to_generate = 10 #@param {type: "number"}
generated_text, probs = generation.generate_text(
    prompt,
    num_tokens_to_generate,
    model=model,
    tokenizer=tokenizer,
    pad_token_id=tokenizer.pad_token_id,
    sampling_mode="greedy" # To generate the highest probability generation.
)

print("Generated text:", generated_text)
print("\n")

visualizations.plot_next_token(probs[0], prompt=prompt, tokenizer=tokenizer)


In [ ]:
prompt = "Jide was hungry so she went looking for" #@param {type: "string"}
num_tokens_to_generate = 10 #@param {type: "number"}
generated_text, probs = generation.generate_text(
    prompt,
    num_tokens_to_generate,
    model=model,
    tokenizer=tokenizer,
    pad_token_id=tokenizer.pad_token_id,
    sampling_mode="random",
)
print("Generated text:", generated_text)
print("\n")

visualizations.plot_next_token(probs[0], prompt=prompt, tokenizer=tokenizer)

In [ ]:
prompt = "Jide was thirsty so she went looking for" #@param {type: "string"}
num_tokens_to_generate = 10 #@param {type: "number"}
generated_text, probs = generation.generate_text(
    prompt,
    num_tokens_to_generate,
    model=model,
    tokenizer=tokenizer,
    pad_token_id=tokenizer.pad_token_id,
    sampling_mode="random",
)

print("Generated text:", generated_text)
print("\n")

visualizations.plot_next_token(probs[0], prompt=prompt, tokenizer=tokenizer)
